In [0]:
base_path = "/Volumes/customer360_dev/staging/source_files"

entities = [
    "customers",
    "addresses",
    "subscriptions",
    "products",
    "orders",
    "order_items",
    "payments",
    "support_tickets"
]

for entity in entities:
    initial_path = f"{base_path}/{entity}/initial"
    incremental_path = f"{base_path}/{entity}/incremental"

    dbutils.fs.mkdirs(initial_path)
    dbutils.fs.mkdirs(incremental_path)

    print(f"Created: {initial_path}")
    print(f"Created: {incremental_path}")

Created: /Volumes/customer360_dev/staging/source_files/customers/initial
Created: /Volumes/customer360_dev/staging/source_files/customers/incremental
Created: /Volumes/customer360_dev/staging/source_files/addresses/initial
Created: /Volumes/customer360_dev/staging/source_files/addresses/incremental
Created: /Volumes/customer360_dev/staging/source_files/subscriptions/initial
Created: /Volumes/customer360_dev/staging/source_files/subscriptions/incremental
Created: /Volumes/customer360_dev/staging/source_files/products/initial
Created: /Volumes/customer360_dev/staging/source_files/products/incremental
Created: /Volumes/customer360_dev/staging/source_files/orders/initial
Created: /Volumes/customer360_dev/staging/source_files/orders/incremental
Created: /Volumes/customer360_dev/staging/source_files/order_items/initial
Created: /Volumes/customer360_dev/staging/source_files/order_items/incremental
Created: /Volumes/customer360_dev/staging/source_files/payments/initial
Created: /Volumes/custom

In [0]:
for entity in entities:
    path = f"{base_path}/{entity}"
    print(f"\n{entity.upper()}")
    
    for item in dbutils.fs.ls(path):
        print(item.path)


CUSTOMERS
dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/
dbfs:/Volumes/customer360_dev/staging/source_files/customers/initial/

ADDRESSES
dbfs:/Volumes/customer360_dev/staging/source_files/addresses/incremental/
dbfs:/Volumes/customer360_dev/staging/source_files/addresses/initial/

SUBSCRIPTIONS
dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/
dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/initial/

PRODUCTS
dbfs:/Volumes/customer360_dev/staging/source_files/products/incremental/
dbfs:/Volumes/customer360_dev/staging/source_files/products/initial/

ORDERS
dbfs:/Volumes/customer360_dev/staging/source_files/orders/incremental/
dbfs:/Volumes/customer360_dev/staging/source_files/orders/initial/

ORDER_ITEMS
dbfs:/Volumes/customer360_dev/staging/source_files/order_items/incremental/
dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/

PAYMENTS
dbfs:/Volumes/customer360_dev/staging/source_files/pay

Create the Data Generation Configuration
Before generating ~30M records, we should centralize all dataset sizes, file counts, paths, and quality settings in one config.
That way, if later we want to increase Orders from 5M to 10M, we change one place, not every notebook.

In [0]:
BASE_PATH = "/Volumes/customer360_dev/staging/source_files"

DATA_CONFIG = {
    "customers": {
        "total_rows": 1_000_000,
        "num_files": 10,
        "rows_per_file": 100_000
    },

    "addresses": {
        "total_rows": 1_200_000,
        "num_files": 12,
        "rows_per_file": 100_000
    },

    "subscriptions": {
        "total_rows": 1_000_000,
        "num_files": 10,
        "rows_per_file": 100_000
    },

    "products": {
        "total_rows": 150_000,
        "num_files": 3,
        "rows_per_file": 50_000
    },

    "orders": {
        "total_rows": 5_000_000,
        "num_files": 20,
        "rows_per_file": 250_000
    },

    "order_items": {
        "total_rows": 15_000_000,
        "num_files": 30,
        "rows_per_file": 500_000
    },

    "payments": {
        "total_rows": 6_000_000,
        "num_files": 24,
        "rows_per_file": 250_000
    },

    "support_tickets": {
        "total_rows": 1_500_000,
        "num_files": 15,
        "rows_per_file": 100_000
    }
}

In [0]:
GENERATION_CONFIG = {
    "random_seed": 42,

    # Intentionally imperfect source data.
    "duplicate_rate": 0.005,      # 0.5%
    "null_rate": 0.005,           # 0.5%
    "invalid_value_rate": 0.005,  # 0.5%
    "referential_error_rate": 0.002,

    "country": "India",

    "historical_start_date": "2024-01-01",
    "historical_end_date": "2026-09-25"
}

In [0]:
for entity, config in DATA_CONFIG.items():
    config["initial_path"] = f"{BASE_PATH}/{entity}/initial"
    config["incremental_path"] = f"{BASE_PATH}/{entity}/incremental"

In [0]:
for entity, config in DATA_CONFIG.items():
    print(
        f"{entity:20} "
        f"rows={config['total_rows']:,} | "
        f"files={config['num_files']} | "
        f"rows/file={config['rows_per_file']:,}"
    )

customers            rows=1,000,000 | files=10 | rows/file=100,000
addresses            rows=1,200,000 | files=12 | rows/file=100,000
subscriptions        rows=1,000,000 | files=10 | rows/file=100,000
products             rows=150,000 | files=3 | rows/file=50,000
orders               rows=5,000,000 | files=20 | rows/file=250,000
order_items          rows=15,000,000 | files=30 | rows/file=500,000
payments             rows=6,000,000 | files=24 | rows/file=250,000
support_tickets      rows=1,500,000 | files=15 | rows/file=100,000


In [0]:
for entity, config in DATA_CONFIG.items():
    expected_rows = config["num_files"] * config["rows_per_file"]

    if expected_rows != config["total_rows"]:
        raise ValueError(
            f"{entity}: total_rows={config['total_rows']:,} "
            f"but num_files × rows_per_file={expected_rows:,}"
        )

print("All generation configurations are valid.")

All generation configurations are valid.
